In [26]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv("../data/creditwise_raw_data.csv")

In [28]:
labeled_df = df[df["Loan_Approved"].notna()].copy()
unlabeled_df = df[df["Loan_Approved"].isna()].copy()

In [29]:
X = labeled_df.drop(
    columns=["Loan_Approved", "Applicant_ID"]
)

y = labeled_df["Loan_Approved"].map({
    "No": 0,
    "Yes": 1
})

## Train - test split

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
X_train.shape

(760, 18)

## Feature Engineering

In [32]:
def add_features(df):
    df = df.copy()
    
    df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
    df["Credit_Score_sq"] = df["Credit_Score"] ** 2
    df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

    return df

In [33]:
X_train = add_features(X_train)
X_test = add_features(X_test)

In [34]:
X_train.shape

(760, 21)

## Separate numerical and categorical features

In [35]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Numerical columns:")
print(num_cols)

print("\nCategorical columns:")
print(cat_cols)

Numerical columns:
['Applicant_Income', 'Coapplicant_Income', 'Age', 'Dependents', 'Credit_Score', 'Existing_Loans', 'DTI_Ratio', 'Savings', 'Collateral_Value', 'Loan_Amount', 'Loan_Term', 'DTI_Ratio_sq', 'Credit_Score_sq', 'Applicant_Income_log']

Categorical columns:
['Employment_Status', 'Marital_Status', 'Loan_Purpose', 'Property_Area', 'Education_Level', 'Gender', 'Employer_Category']


## Numerical preprocessing

In [36]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

## Categorical preprocessing

In [37]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [38]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
])

In [39]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (760, 30)
Testing shape: (190, 30)


## Naive Bayes pipeline

In [40]:
from sklearn.naive_bayes import GaussianNB

nb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GaussianNB())
])

nb_pipeline.fit(X_train, y_train)

y_pred_nb = nb_pipeline.predict(X_test)

y_prob_nb = nb_pipeline.predict_proba(X_test)[:, 1]

In [41]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Naive Bayes Pipeline")
print("-------------------------")

print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))
print("F1 Score:", f1_score(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Naive Bayes Pipeline
-------------------------
Precision: 0.847457627118644
Recall: 0.8333333333333334
F1 Score: 0.8403361344537815
Accuracy: 0.9
ROC-AUC: 0.9632051282051282

Confusion Matrix:
[[121   9]
 [ 10  50]]


## cross-validation

In [42]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [43]:
nb_cv_accuracy = cross_val_score(
    nb_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy"
)

print("CV Accuracy:", nb_cv_accuracy)
print("Mean Accuracy:", nb_cv_accuracy.mean())
print("Std Accuracy:", nb_cv_accuracy.std())

CV Accuracy: [0.92105263 0.90131579 0.88815789 0.88815789 0.91447368]
Mean Accuracy: 0.9026315789473683
Std Accuracy: 0.013418472404191518


In [44]:
nb_cv_auc = cross_val_score(
    nb_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring="roc_auc"
)

print("CV ROC-AUC:", nb_cv_auc)
print("Mean ROC-AUC:", nb_cv_auc.mean())
print("Std:", nb_cv_auc.std())

CV ROC-AUC: [0.96674679 0.96213942 0.96654647 0.95785208 0.97325228]
Mean ROC-AUC: 0.9653074097887929
Std: 0.005144826412749989


## Hyperparameter tuning

In [45]:
from sklearn.linear_model import LogisticRegression

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [46]:
from sklearn.model_selection import GridSearchCV

log_grid = GridSearchCV(
    log_pipeline,
    param_grid,
    cv=skf,
    scoring="roc_auc",
    n_jobs=1
)

log_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Applicant_Income',
                                                                          'Coapplicant_Income',
                                                                          'Age',
                                                                          'Dependents',
                                                                          'Credit_Score',
                                                                          'Existing_Loans',
                                                                          'DTI_Ratio',
                                                                          'Savings',
                                                                          'Collate...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore',
                                                                                                        sparse_output=False))]),
                                                                         ['Employment_Status',
                                                                          'Marital_Status',
                                                                          'Loan_Purpose',
                                                                          'Property_Area',
                                                                          'Education_Level',
                                                                          'Gender',
                                                                          'Employer_Category'])])),
                                       ('model',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=1, param_grid={'model__C': [0.01, 0.1, 1, 10, 100]},
             scoring='roc_auc')

In [47]:
print("Best Parameters:", log_grid.best_params_)
print("Best CV ROC-AUC:", log_grid.best_score_)

Best Parameters: {'model__C': 100}
Best CV ROC-AUC: 0.9688133475566986


In [48]:
param_grid_nb = {
    "model__var_smoothing": [
        1e-11,
        1e-10,
        1e-9,
        1e-8,
        1e-7,
        1e-6,
        1e-5
    ]
}

nb_grid = GridSearchCV(
    nb_pipeline,
    param_grid_nb,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1
)

nb_grid.fit(X_train, y_train)

print("Best Parameters:", nb_grid.best_params_)
print("Best CV ROC-AUC:", nb_grid.best_score_)

Best Parameters: {'model__var_smoothing': 1e-11}
Best CV ROC-AUC: 0.9653074097887929


## final test-set comparison

In [49]:
best_log_model = log_grid.best_estimator_

y_pred_log = best_log_model.predict(X_test)
y_prob_log = best_log_model.predict_proba(X_test)[:, 1]

print("Tuned Logistic Regression")
print("-------------------------")
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))
print("F1 Score:", f1_score(y_test, y_pred_log))
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

Tuned Logistic Regression
-------------------------
Precision: 0.8846153846153846
Recall: 0.7666666666666667
F1 Score: 0.8214285714285714
Accuracy: 0.8947368421052632
ROC-AUC: 0.9674358974358974

Confusion Matrix:
[[124   6]
 [ 14  46]]


In [50]:
best_nb_model = nb_grid.best_estimator_

y_pred_nb = best_nb_model.predict(X_test)
y_prob_nb = best_nb_model.predict_proba(X_test)[:, 1]

print("Tuned Naive Bayes")
print("-------------------------")
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))
print("F1 Score:", f1_score(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Tuned Naive Bayes
-------------------------
Precision: 0.847457627118644
Recall: 0.8333333333333334
F1 Score: 0.8403361344537815
Accuracy: 0.9
ROC-AUC: 0.9632051282051282

Confusion Matrix:
[[121   9]
 [ 10  50]]


## Feature Selection

In [57]:
# Get the best Logistic Regression pipeline
best_log_model = log_grid.best_estimator_

# Get the fitted preprocessor
fitted_preprocessor = best_log_model.named_steps["preprocessor"]

# Get transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = best_log_model.named_steps["model"].coef_[0]

# Create a dataframe
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Abs_Coefficient": np.abs(coefficients)
})

# Sort by importance
feature_importance = feature_importance.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

feature_importance

,Feature,Coefficient,Abs_Coefficient
4,num__Credit_Score,26.344598,26.344598
12,num__Credit_Score_sq,-23.041170,23.041170
11,num__DTI_Ratio_sq,-11.423081,11.423081
6,num__DTI_Ratio,6.913621,6.913621
13,num__Applicant_Income_log,4.294977,4.294977
0,num__Applicant_Income,-3.249446,3.249446
22,cat__Property_Area_Semiurban,1.520693,1.520693
23,cat__Property_Area_Urban,0.961996,0.961996
27,cat__Employer_Category_MNC,0.915323,0.915323
16,cat__Employment_Status_Unemployed,-0.813330,0.813330


In [58]:
# Feature selection experiment

features_to_remove = [
    "Savings",
    "Collateral_Value",
    "Existing_Loans",
    "Marital_Status",
    "Loan_Term",
    "Age",
    "Dependents"
]

X_train_selected = X_train.drop(columns=features_to_remove)
X_test_selected = X_test.drop(columns=features_to_remove)

X_train_selected.shape, X_test_selected.shape

((760, 14), (190, 14))

In [59]:
cat_cols_selected = X_train_selected.select_dtypes(
    include=["object"]
).columns.tolist()

num_cols_selected = X_train_selected.select_dtypes(
    include=["number"]
).columns.tolist()

In [60]:
preprocessor_selected = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]), num_cols_selected),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]), cat_cols_selected)
])

log_selected_pipeline = Pipeline([
    ("preprocessor", preprocessor_selected),
    ("model", LogisticRegression(
        C=100,
        max_iter=1000
    ))
])

selected_cv_scores = cross_val_score(
    log_selected_pipeline,
    X_train_selected,
    y_train,
    cv=skf,
    scoring="roc_auc"
)

print("Selected Feature CV ROC-AUC:")
print(selected_cv_scores)

print("Mean CV ROC-AUC:", selected_cv_scores.mean())
print("Std CV ROC-AUC:", selected_cv_scores.std())

Selected Feature CV ROC-AUC:
[0.9765625  0.96814904 0.96434295 0.96595745 0.98115502]
Mean CV ROC-AUC: 0.9712333898371132
Std CV ROC-AUC: 0.00650638212814388


In [61]:
log_selected_pipeline.fit(
    X_train_selected,
    y_train
)

y_pred_selected = log_selected_pipeline.predict(
    X_test_selected
)

y_prob_selected = log_selected_pipeline.predict_proba(
    X_test_selected
)[:, 1]

print("Selected Feature Logistic Regression")
print("------------------------------------")

print("Precision:", precision_score(y_test, y_pred_selected))
print("Recall:", recall_score(y_test, y_pred_selected))
print("F1 Score:", f1_score(y_test, y_pred_selected))
print("Accuracy:", accuracy_score(y_test, y_pred_selected))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_selected))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_selected))

Selected Feature Logistic Regression
------------------------------------
Precision: 0.9038461538461539
Recall: 0.7833333333333333
F1 Score: 0.8392857142857143
Accuracy: 0.9052631578947369
ROC-AUC: 0.9669230769230768

Confusion Matrix:
[[125   5]
 [ 13  47]]


## final 5-fold CV comparison.

In [62]:
lr_final_cv = cross_val_score(
    log_selected_pipeline,
    X_train_selected,
    y_train,
    cv=skf,
    scoring="roc_auc"
)

print("Selected Logistic Regression")
print("Fold ROC-AUC:", lr_final_cv)
print("Mean ROC-AUC:", lr_final_cv.mean())
print("Std ROC-AUC:", lr_final_cv.std())

Selected Logistic Regression
Fold ROC-AUC: [0.9765625  0.96814904 0.96434295 0.96595745 0.98115502]
Mean ROC-AUC: 0.9712333898371132
Std ROC-AUC: 0.00650638212814388


In [63]:
nb_final_cv = cross_val_score(
    nb_grid.best_estimator_,
    X_train,
    y_train,
    cv=skf,
    scoring="roc_auc"
)

print("Tuned Naive Bayes")
print("Fold ROC-AUC:", nb_final_cv)
print("Mean ROC-AUC:", nb_final_cv.mean())
print("Std ROC-AUC:", nb_final_cv.std())

Tuned Naive Bayes
Fold ROC-AUC: [0.96674679 0.96213942 0.96654647 0.95785208 0.97325228]
Mean ROC-AUC: 0.9653074097887929
Std ROC-AUC: 0.005144826412749989
